### Wichteln – Lösung Schritt für Schritt

*39. Bundeswettbewerb Informatik (2020), 1. Runde, Aufgabe 5*

Pirmins Klasse möchte dieses Jahr zu Weihnachten eine besondere Variante des Wichtelns durchführen. Dabei bringt jeder genau
einen Gegenstand mit. Alle Gegenstände werden nummeriert und auf einer Theke mit ihrer jeweiligen Nummer ausgestellt.
Anschließend notiert jeder bzw. jede auf einem Zettel, welcher der ausgestellten Gegenstände ihm oder ihr am besten, am
zweit- bzw. am drittbesten gefällt. Die Zettel werden dann gesammelt.

In der letzten Informatikstunde vor Weihnachten sollen die Gegenstände möglichst gut an die Schülerinnen und Schüler verteilt
werden. Eine Verteilung ist besser als eine andere Verteilung, wenn die Anzahl der ersten Wünsche, die erfüllt werden,
höher ist als bei der anderen Verteilung. Ist diese Anzahl bei beiden Verteilungen gleich, entscheidet die Anzahl der
zweiten Wünsche, die erfüllt sind. Ist auch diese Anzahl gleich, ist die Anzahl der erfüllten dritten Wünsche
ausschlaggebend. Ist diese ebenfalls gleich, gelten die Verteilungen als gleich gut.

#### Aufgabe 5

Schreibe ein Programm, das solche Wünsche von Schülerinnen und Schülern einliest und eine möglichst gute Verteilung gemäß
der obigen Definition ausgibt.

Wende dein Programm auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/wichteln1.txt` bis `wichteln7.txt`)

1. Zeile: Anzahl n der Schülerinnen und Schüler (Schüler i bringt Gegenstand i mit); dann je Zeile die drei Wünsche
(erster, zweiter, dritter) von Schüler 1, 2, …

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zuordnungsproblem (Matching):** Personen und Geschenke bilden zwei Gruppen. Jede Person bekommt genau ein Geschenk, jedes Geschenk geht an genau
  eine Person. Eine Wunschliste ist eine Menge erlaubter Paare, die wir mit Punkten bewerten.
- **Lexikografisch optimieren durch Punkte:** Ein Erstwunsch ist mehr wert als *alle* Zweitwünsche zusammen, ein Zweitwunsch mehr als *alle*
  Drittwünsche zusammen. Mit passenden Punktzahlen wird aus der Reihenfolge der Aufgabe eine einzige Zahl, die man maximiert.
- **Verbessernde Wege:** Eine bestehende Zuordnung lässt sich verbessern, indem man eine Kette „A nimmt das Geschenk von B, B nimmt das Geschenk von C, …“
  ausführt. Das ist die Grundidee der Ungarischen Methode und der Flussalgorithmen.
- **Warteschlange (Queue):** Die Suche nach dem besten Weg arbeitet eine Warteschlange ab.
- **Greedy-Verfahren als Vergleich:** Ein einfaches gieriges Verfahren ist schnell geschrieben, zeigt aber, warum man mehr braucht.

**Nützliche Funktionen und Bibliotheken**
- `collections.deque` mit `append` und `popleft` für die Warteschlange.
- Wörterbücher (`dict`), hier: `wert[p][g]` = Punkte, die Person `p` für Geschenk `g` bekommt.
- `liste.index(x)` liefert die Position (also den Rang) eines Wunsches in der Wunschliste.

## Die Idee in Worten

**Punkte statt Reihenfolge.** Bei `n` Personen vergeben wir

- `n² + n + 1` Punkte für einen erfüllten Erstwunsch,
- `n + 1` Punkte für einen erfüllten Zweitwunsch,
- `1` Punkt für einen erfüllten Drittwunsch.

Warum genügt das? Höchstens `n` Zweitwünsche sind erfüllbar, zusammen also höchstens `n · (n + 1) = n² + n` Punkte, weniger als ein einziger Erstwunsch (`n² + n + 1`).
Ebenso sind `n` Drittwünsche zusammen `n` Punkte und damit weniger als ein Zweitwunsch (`n + 1`). Eine Verteilung mit mehr Erstwünschen hat deshalb immer mehr Punkte,
bei gleich vielen Erstwünschen entscheiden die Zweitwünsche, dann die Drittwünsche. **Die beste Verteilung im Sinne der Aufgabe ist genau die mit den meisten Punkten.**

**Wunschpaare mit Punkten sammeln.** Wir bauen für jede Person ein Wörterbuch `wert[p]`, das jedem gewünschten Geschenk seine Punkte zuordnet. Nennt eine Person ein Geschenk
mehrfach, zählt der beste Rang. Geschenke, die nicht auf dem Zettel stehen, bekommen keine Punkte und kommen gar nicht erst ins Wörterbuch.

**Verteilung verbessern.** Wir beginnen ohne Zuordnung und fügen so lange *verbessernde Wege* ein, wie dadurch die Punkte steigen. Ein Weg beginnt bei einer Person ohne Geschenk `p0`.
`p0` nimmt ein Geschenk `g1`. Gehört `g1` schon `p1`, muss `p1` auf ein anderes Geschenk `g2` ausweichen und so weiter, bis die Kette bei einem noch freien Geschenk endet.
Der Gewinn des Weges ist die Summe der Punkte der neuen Paare minus die Summe der Punkte der aufgelösten Paare. Wir nehmen jedes Mal den Weg mit dem größten Gewinn,
solange er größer als 0 ist. Dass man so die optimale Verteilung erhält, ist ein bekannter Satz zu gewichteten Matchings. Wir prüfen es unten an den Zahlen der Lösungshinweise.

**Rest verteilen.** Wer danach kein Geschenk hat, bekommt eines der übrig gebliebenen Geschenke. Das kann keinen Wunsch mehr erfüllen: Hätte eine solche Person einen Wunsch, der noch frei ist,
gäbe es einen Weg mit positivem Gewinn.

## Schritt 1: Einlesen und Punkte

Die Datei enthält zuerst `n`, dann `3 · n` Zahlen. Wir lesen alles als Zahlenliste. Person `p` (ab 0) hat die Wünsche an den Positionen `3p+1`, `3p+2`, `3p+3`.
Wir rechnen die Nummern auf 0 bis `n-1` um und kehren erst bei der Ausgabe zu 1 bis `n` zurück.

In [ ]:
def lies_wuensche(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zahlen = [int(z) for z in f.read().split()]
    n = zahlen[0]
    wuensche = []
    for p in range(n):
        wunsch1 = zahlen[1 + 3 * p] - 1
        wunsch2 = zahlen[2 + 3 * p] - 1
        wunsch3 = zahlen[3 + 3 * p] - 1
        wuensche.append([wunsch1, wunsch2, wunsch3])
    return wuensche


def baue_punkte(wuensche):
    """wert[p][g] = Punkte, die Person p für Geschenk g bekommt (nur gewünschte Geschenke)."""
    n = len(wuensche)
    punkte_je_rang = [n * n + n + 1, n + 1, 1]
    wert = []
    for p in range(n):
        eintraege = {}
        for rang in range(3):
            g = wuensche[p][rang]
            if g not in eintraege:                  # bei doppelter Nennung zählt der bessere Rang
                eintraege[g] = punkte_je_rang[rang]
        wert.append(eintraege)
    return wert


wuensche1 = lies_wuensche('beispieldaten/wichteln1.txt')
print(wuensche1[:3])
print(baue_punkte(wuensche1)[:3])

[[1, 9, 5], [1, 6, 2], [3, 6, 0]]
[{1: 111, 9: 11, 5: 1}, {1: 111, 6: 11, 2: 1}, {3: 111, 6: 11, 0: 1}]


Bei `n = 10` sind das 111, 11 und 1 Punkt. Person 1 wünscht sich die Geschenke 2, 10 und 6 (in der Zählung ab 0 sind das 1, 9 und 5), und genau diese drei Einträge stehen in `wert[0]`.

## Schritt 2: Einen verbessernden Weg finden

`pg[p]` ist das Geschenk von Person `p` (oder `-1`), `gp[g]` die Person, die Geschenk `g` hat (oder `-1`). Die Suche läuft in einer Warteschlange über *Personen*:

- Alle Personen **ohne** Geschenk starten mit Gewinn 0.
- Für eine Person `p` und jedes ihrer Wunschgeschenke `g` (das sie noch nicht hat) ist `gewinn_p[p] + wert[p][g]` der Gewinn, wenn `p` das Geschenk `g` nimmt.
  Ist das mehr als der bisher beste Gewinn für `g`, merken wir ihn (`gewinn_g[g]`) und `p` als Vorgänger (`von_person[g]`).
- Gehört `g` schon einer Person `q`, muss `q` ausweichen. Der Gewinn für `q` ist der Gewinn für `g` minus die Punkte, die `q` an `g` verliert. Auch das wird gemerkt,
  und `q` kommt in die Warteschlange.
- Am Ende suchen wir unter den **freien** Geschenken das mit dem größten Gewinn. Ist er größer als 0, führen wir den Weg rückwärts über die gemerkten Vorgänger aus.

Weil die Zuordnung vor jeder Runde schon optimal für die bisherigen Personen ist, gibt es keine „Kreise mit Gewinn“, und die Suche endet.

In [ ]:
from collections import deque

KEINER = -10 ** 18          # „noch nicht erreicht“


def verbessere_einmal(wert, pg, gp):
    """Sucht den besten verbessernden Weg und führt ihn aus. Gibt (gefunden, Zahl der Prüfungen) zurück."""
    n = len(wert)
    gewinn_p = [KEINER] * n         # bester Gewinn, mit dem Person p ihr altes Geschenk loswird und neu wählt
    gewinn_g = [KEINER] * n         # bester Gewinn, wenn Geschenk g in die Hände einer Person kommt
    von_person = [-1] * n           # von_person[g]: Person, die g im besten Weg nimmt
    schlange = deque()
    for p in range(n):
        if pg[p] == -1:
            gewinn_p[p] = 0
            schlange.append(p)
    pruefungen = 0
    while len(schlange) > 0:
        p = schlange.popleft()
        for g in wert[p]:
            pruefungen += 1
            if pg[p] == g:
                continue                                    # dieses Geschenk hat p schon
            neu = gewinn_p[p] + wert[p][g]
            if neu > gewinn_g[g]:
                gewinn_g[g] = neu
                von_person[g] = p
                q = gp[g]
                if q != -1:                                 # q muss g abgeben und sucht ein neues
                    neu_q = neu - wert[q][g]
                    if neu_q > gewinn_p[q]:
                        gewinn_p[q] = neu_q
                        schlange.append(q)
    bestes = -1
    for g in range(n):
        if gp[g] == -1 and gewinn_g[g] > 0:
            if bestes == -1 or gewinn_g[g] > gewinn_g[bestes]:
                bestes = g
    if bestes == -1:
        return False, pruefungen
    g = bestes
    while g != -1:                                          # Weg rückwärts ausführen
        p = von_person[g]
        altes = pg[p]
        pg[p] = g
        gp[g] = p
        g = altes
    return True, pruefungen

Beim Ausführen des Weges geht es von hinten nach vorn: Die letzte Person nimmt das freie Geschenk, ihr altes Geschenk (`altes`) ist jetzt frei und wird von der
Person davor genommen und so weiter, bis die Person ohne Geschenk erreicht ist (ihr `altes` ist `-1`, die Schleife endet).

## Schritt 3: Alles zusammen

Wir wiederholen die Suche, bis kein Weg mehr Gewinn bringt. Danach bekommen Personen ohne Geschenk die übrig gebliebenen Geschenke.
Zur Ausgabe gehört, welcher Wunsch erfüllt wurde: `wuensche[p].index(g)` ist der Rang (0, 1 oder 2) des Geschenks in der Liste von `p`.

In [ ]:
def verteile(wuensche):
    n = len(wuensche)
    wert = baue_punkte(wuensche)
    pg = [-1] * n
    gp = [-1] * n
    runden = 0
    gesamt = 0
    weiter = True
    while weiter:
        weiter, pruefungen = verbessere_einmal(wert, pg, gp)
        gesamt += pruefungen
        if weiter:
            runden += 1
    uebrig = [g for g in range(n) if gp[g] == -1]
    for p in range(n):
        if pg[p] == -1:
            g = uebrig.pop()
            pg[p] = g
            gp[g] = p
    return pg, runden, gesamt


def zaehle_wuensche(wuensche, pg):
    """[erste, zweite, dritte] = Zahl der erfüllten Wünsche je Rang."""
    z = [0, 0, 0]
    for p in range(len(wuensche)):
        if pg[p] in wuensche[p]:
            z[wuensche[p].index(pg[p])] += 1
    return z


def zeige(wuensche, pg):
    namen = ['1. Wunsch', '2. Wunsch', '3. Wunsch']
    for p in range(len(wuensche)):
        text = 'Kind %d erhält Geschenk %d' % (p + 1, pg[p] + 1)
        if pg[p] in wuensche[p]:
            text += ' (%s)' % namen[wuensche[p].index(pg[p])]
        print(text)


pg1, runden1, pruefungen1 = verteile(wuensche1)
zeige(wuensche1, pg1)
print('Erfüllt (1., 2., 3. Wunsch):', zaehle_wuensche(wuensche1, pg1))

Kind 1 erhält Geschenk 2 (1. Wunsch)
Kind 2 erhält Geschenk 8
Kind 3 erhält Geschenk 1 (3. Wunsch)
Kind 4 erhält Geschenk 3 (1. Wunsch)
Kind 5 erhält Geschenk 5
Kind 6 erhält Geschenk 4 (1. Wunsch)
Kind 7 erhält Geschenk 7 (1. Wunsch)
Kind 8 erhält Geschenk 10 (1. Wunsch)
Kind 9 erhält Geschenk 9 (1. Wunsch)
Kind 10 erhält Geschenk 6 (3. Wunsch)
Erfüllt (1., 2., 3. Wunsch): [6, 0, 2]


**Probe und alle Beispiele:** Jedes Geschenk darf genau einmal vorkommen. Das prüft die nächste Zelle. Gleichzeitig zählen wir die erfüllten Wünsche aller sieben Beispiele
sowie die Zahl der Wege und der Prüfungen, damit man den Aufwand sieht (Zeiten hängen vom Rechner ab, Zahlen nicht).

In [ ]:
ERGEBNISSE = {}
print('Datei      n  erste zweite dritte   Wege  Prüfungen')
for nr in range(1, 8):
    wuensche = lies_wuensche('beispieldaten/wichteln%d.txt' % nr)
    pg, runden, pruefungen = verteile(wuensche)
    assert sorted(pg) == list(range(len(wuensche)))         # jedes Geschenk genau einmal
    z = zaehle_wuensche(wuensche, pg)
    ERGEBNISSE[nr] = z
    print('%5d %6d %6d %6d %6d %6d %10d' % (nr, len(wuensche), z[0], z[1], z[2], runden, pruefungen))

Datei      n  erste zweite dritte   Wege  Prüfungen
    1     10      6      0      2      8        309
    2     10      3      0      0      3        135
    3     30     15      6      1     22       2328
    4     30     15      4      3     22       2133
    5     30     13      1      7     21       2244
    6     90     37      3     21     61      17532
    7   1000    541    127     58    726    2354604


Die Zahlen stimmen mit Tabelle 15 der offiziellen Lösungshinweise überein (Beispiele 1 bis 7: 6/0/2, 3/0/0, 15/6/1, 15/4/3, 13/1/7, 37/3/21, 541/127/58).
Beispiel 7 mit 1000 Personen braucht rund 2,4 Millionen Prüfungen und ist trotzdem schnell fertig.

Zur Einordnung: Die Hinweise lösen dasselbe Problem mit der Ungarischen Methode (Laufzeit proportional zu `n³`). Unser Verfahren nutzt aus, dass jede Person nur drei Wunschpaare hat.
Ein Weg kostet deshalb nur einige tausend Prüfungen statt `n²`.

## Vergleich: ein einfaches Greedy-Verfahren

Der erste Gedanke: Erst alle Erstwünsche verteilen, dann alle Zweitwünsche, dann alle Drittwünsche, und zwar jeweils so, dass ein Geschenk, das noch frei ist,
an die nächste Person geht, die es sich wünscht. Wir probieren das aus, um zu sehen, wie weit man damit kommt.

In [ ]:
def greedy(wuensche):
    n = len(wuensche)
    pg = [-1] * n
    gp = [-1] * n
    for rang in range(3):
        for p in range(n):
            g = wuensche[p][rang]
            if pg[p] == -1 and gp[g] == -1:
                pg[p] = g
                gp[g] = p
    uebrig = [g for g in range(n) if gp[g] == -1]
    for p in range(n):
        if pg[p] == -1:
            g = uebrig.pop()
            pg[p] = g
            gp[g] = p
    return pg


print('Datei  Greedy (1., 2., 3.)   Optimum (1., 2., 3.)')
GREEDY = {}
for nr in range(1, 8):
    wuensche = lies_wuensche('beispieldaten/wichteln%d.txt' % nr)
    zg = zaehle_wuensche(wuensche, greedy(wuensche))
    GREEDY[nr] = zg
    print('%5d  %-21s %s' % (nr, zg, ERGEBNISSE[nr]))

Datei  Greedy (1., 2., 3.)   Optimum (1., 2., 3.)
    1  [6, 0, 1]             [6, 0, 2]
    2  [3, 0, 0]             [3, 0, 0]
    3  [15, 6, 1]            [15, 6, 1]
    4  [15, 3, 3]            [15, 4, 3]
    5  [13, 1, 6]            [13, 1, 7]
    6  [37, 3, 18]           [37, 3, 21]
    7  [541, 94, 50]         [541, 127, 58]


Das Greedy-Verfahren erreicht bei den Erstwünschen schon das Optimum, bleibt aber bei den Beispielen 1, 4, 5, 6 und 7 bei den Zweit- und Drittwünschen dahinter
(bei Beispiel 7 sind es 94 statt 127 Zweitwünsche und 50 statt 58 Drittwünsche). Greedy kann nicht „zurückrudern“: Wer früh dran ist, nimmt einem anderen
das Geschenk weg, auch wenn er selbst noch ein anderes gewünschtes Geschenk hätte. Genau das leisten die verbessernden Wege.
Die Lösungshinweise erwähnen als gute Heuristik, einzigartige Erstwünsche zuerst zu vergeben.

## Für Neugierige: einzigartige Erstwünsche

Wünscht sich nur *eine* Person ein bestimmtes Geschenk als erstes, bekommt sie es in jeder optimalen Verteilung (sonst könnte man tauschen und gewänne einen Erstwunsch).
Das Problem schrumpft dann, bevor man überhaupt sucht. Wir zählen sie in den Beispielen und prüfen, dass unsere Verteilung sie wirklich erfüllt:

In [ ]:
def einzigartige_erstwuensche(wuensche):
    haeufigkeit = {}
    for w in wuensche:
        haeufigkeit[w[0]] = haeufigkeit.get(w[0], 0) + 1
    return [p for p in range(len(wuensche)) if haeufigkeit[wuensche[p][0]] == 1]


for nr in range(1, 8):
    wuensche = lies_wuensche('beispieldaten/wichteln%d.txt' % nr)
    pg, runden, pruefungen = verteile(wuensche)
    einzig = einzigartige_erstwuensche(wuensche)
    erfuellt = sum(1 for p in einzig if pg[p] == wuensche[p][0])
    print('Datei %d: %d einzigartige Erstwünsche, davon erfüllt: %d' % (nr, len(einzig), erfuellt))

Datei 1: 3 einzigartige Erstwünsche, davon erfüllt: 3
Datei 2: 0 einzigartige Erstwünsche, davon erfüllt: 0
Datei 3: 5 einzigartige Erstwünsche, davon erfüllt: 5
Datei 4: 9 einzigartige Erstwünsche, davon erfüllt: 9
Datei 5: 5 einzigartige Erstwünsche, davon erfüllt: 5
Datei 6: 14 einzigartige Erstwünsche, davon erfüllt: 14
Datei 7: 296 einzigartige Erstwünsche, davon erfüllt: 296


## Zum Schluss: Wichteln mit MiniZinc

Die Aufgabe ist ein Optimierungsproblem mit Zuordnungsbedingungen und passt daher gut zu MiniZinc. Wir verwenden dieselben Punkte wie oben (`n² + n + 1`, `n + 1`, `1`).

**Modell.** Statt für jede Person das Geschenk als Variable zu wählen, gibt es für jede Person und jeden ihrer drei Wünsche eine 0/1-Variable `nimm[p, r]`:
1 bedeutet „Person `p` bekommt ihren `r`-ten Wunsch“. Zwei Bedingungen genügen:

- jede Person bekommt höchstens einen Wunsch (`sum … <= 1`),
- jedes Geschenk geht an höchstens eine Person (`sum … where wunsch[p, r] == g`, dabei zählen nur die Wünsche, die auf `g` zeigen).

Wer keinen Wunsch bekommt, erhält später ein übrig gebliebenes Geschenk. Wir maximieren die Punkte, und die Ausgabe zählt die erfüllten Wünsche je Rang.

```minizinc
int: n;
array[1..n, 1..3] of 1..n: wunsch;          % wunsch[p, r]: der r-te Wunsch von Person p
array[1..n, 1..3] of var 0..1: nimm;        % nimm[p, r] = 1: Person p bekommt ihren r-ten Wunsch

% jede Person bekommt höchstens einen ihrer Wünsche
constraint forall(p in 1..n)(sum(r in 1..3)(nimm[p, r]) <= 1);
% jedes Geschenk geht an höchstens eine Person
constraint forall(g in 1..n)(sum(p in 1..n, r in 1..3 where wunsch[p, r] == g)(nimm[p, r]) <= 1);

array[1..3] of int: punkte = [n * n + n + 1, n + 1, 1];
solve maximize sum(p in 1..n, r in 1..3)(punkte[r] * nimm[p, r]);

output ["erste: ", show(sum(p in 1..n)(nimm[p, 1])),
        ", zweite: ", show(sum(p in 1..n)(nimm[p, 2])),
        ", dritte: ", show(sum(p in 1..n)(nimm[p, 3])), "\n"];
```

**Daten.** Die Datei `wichteln1.txt` wird zu einer `.dzn`-Datei. Die Python-Zelle `als_dzn` dazu steht unten. Für Beispiel 1 sieht die Datei so aus:

```minizinc
n = 10;
wunsch = array2d(1..n, 1..3, [2, 10, 6, 2, 7, 3, 4, 7, 1, 3, 4, 9, 3, 7, 9, 4, 3, 2, 7, 6, 2, 10, 2, 4, 9, 8, 1, 4, 9, 6]);
```

Ergebnisse für alle sieben Beispiele (Solver **HiGHS**, jedes Beispiel in höchstens ein paar Sekunden):

```
wichteln1.txt  ->  erste: 6, zweite: 0, dritte: 2
wichteln2.txt  ->  erste: 3, zweite: 0, dritte: 0
wichteln3.txt  ->  erste: 15, zweite: 6, dritte: 1
wichteln4.txt  ->  erste: 15, zweite: 4, dritte: 3
wichteln5.txt  ->  erste: 13, zweite: 1, dritte: 7
wichteln6.txt  ->  erste: 37, zweite: 3, dritte: 21
wichteln7.txt  ->  erste: 541, zweite: 127, dritte: 58
```

Das sind genau die Zahlen unserer Python-Lösung und der Lösungshinweise.

**Warum dieses Modell?** Der naheliegende Versuch ist `array[1..n] of var 1..n: x` mit `alldifferent(x)` und dem Ziel
`sum(...(x[p] == wunsch[p, r]) ...)`. Das ist kürzer, aber für den Solver viel schwerer: Mit CP-SAT sind die Beispiele 1 bis 5 in etwa zwei Sekunden
fertig, bei Beispiel 6 (90 Personen) und Beispiel 7 gibt es auch nach einer halben Minute keine bewiesene beste Lösung. Chuffed und Gecode sind schon bei Beispiel 3 zu langsam.
Das 0/1-Modell dagegen ist ein Zuordnungsproblem, bei dem die Lösung der LP-Relaxation (ohne die Forderung 0/1) von selbst ganzzahlig wird. Ein MIP-Solver wie HiGHS löst es deshalb ohne lange Suche.
Die Wahl des Modells entscheidet hier über Sekunden oder gar keine Lösung.

In [ ]:
def als_dzn(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zahlen = [int(z) for z in f.read().split()]
    return 'n = %d;\nwunsch = array2d(1..n, 1..3, %s);' % (zahlen[0], zahlen[1:])


print(als_dzn('beispieldaten/wichteln1.txt'))

n = 10;
wunsch = array2d(1..n, 1..3, [2, 10, 6, 2, 7, 3, 4, 7, 1, 3, 4, 9, 3, 7, 9, 4, 3, 2, 7, 6, 2, 10, 2, 4, 9, 8, 1, 4, 9, 6]);
